# Agent 6 — Career Agent: uploaded outcomes and unsupervised learning

**This is the single final notebook. Run every cell from top to bottom.**
The earlier advisory functions and independent-label evaluation utilities are retained;
the appended outcome extension sets the final `career_agent` entry point.

Input: Agent 3's exact university/program identities, student career goal and optional
program ID/UNITID. Data: `data/raw/agent6/agent_6_dataset_final_merged.csv` plus the
Agent 2 catalog for exact program identity. Output: `state["careers"]` with career-path
advice, goal alignment, role examples, cohort earnings, outcome clusters, evidence scope
and placement context. Input order and the caller's state are preserved.

**Learning components:** supervised HistGradientBoosting regression for uploaded four-year
cohort earnings; unsupervised KMeans for log earnings/debt patterns. Institutions are
separated into train/dev/test. Cluster count is selected on dev only; frozen models are
then evaluated on withheld institutions. Estimated program rows and heuristic job/research
scores are excluded from training targets. Saved joblib models retain preprocessing.

The dataset has no independently labelled student research/industry outcomes. Career-path
advice remains explicit rules; a cluster is not a career class. US federal-aid cohort
figures are not international-student salary forecasts. School-wide MBA placement reports
are context, never specialization placement rates. Uploaded source claims are not
independently verified against every original source in this run.

Final measurements and artifact paths appear in the **last cell**. Model files are saved
under `backend/app/agents/career_agent/trained_models/`. Load only trusted joblib files.
Dependencies: numpy, pandas, scipy, scikit-learn, joblib, threadpoolctl.

References: [College Scorecard field-of-study methodology](https://collegescorecard.ed.gov/assets/FieldOfStudyDataDocumentation.pdf),
[HistGradientBoostingRegressor](https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.HistGradientBoostingRegressor.html),
[KMeans silhouette evaluation](https://scikit-learn.org/stable/auto_examples/cluster/plot_kmeans_silhouette_analysis).

## Retained advisory implementation and its original behavior checks


## 1. Local setup and uploaded program catalog

In [1]:
from pathlib import Path
from copy import deepcopy
import hashlib
import json
import math
import re
import numpy as np
import pandas as pd

PROJECT_ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'data/raw').is_dir())
AGENT6_DATA = PROJECT_ROOT / 'data/raw/agent6'
AGENT6_CATALOG_PATH = PROJECT_ROOT / 'data/raw/agent2/agent2_enhanced_program_catalog.csv'
AGENT6_OUTCOMES_PATH = AGENT6_DATA / 'agent6_career_outcomes.csv'
AGENT6_PLACEMENTS_PATH = AGENT6_DATA / 'agent6_placement_reports.csv'
AGENT6_RUNS = AGENT6_DATA / 'evaluation_runs'

def career_text(value):
    if value is None:
        return ''
    if isinstance(value, (list, tuple)):
        return ' '.join(career_text(item) for item in value)
    if isinstance(value, dict):
        return ''
    return '' if pd.isna(value) else str(value).strip()

def career_key(value):
    return ' '.join(career_text(value).casefold().split())

def load_career_catalog(path=AGENT6_CATALOG_PATH):
    frame = pd.read_csv(path).fillna('')
    required = {'university_name', 'program_name', 'description_text', 'curriculum_highlights', 'faculty_research_areas'}
    if required - set(frame):
        raise ValueError('Program catalog is missing descriptive fields.')
    frame['_university_key'] = frame.university_name.map(career_key)
    frame['_program_key'] = frame.program_name.map(career_key)
    if frame[['_university_key', '_program_key']].eq('').any().any():
        raise ValueError('Every catalog row needs a named university and program.')
    # Prefer the richest descriptive duplicate, as in Agent 3.
    frame['_length'] = frame[['description_text', 'curriculum_highlights', 'faculty_research_areas']].agg(' '.join, axis=1).str.len()
    return frame.sort_values('_length', ascending=False, kind='stable').drop_duplicates(
        ['_university_key', '_program_key']).reset_index(drop=True)

career_catalog = load_career_catalog()
print(f'Agent 6 catalog: {len(career_catalog):,} programs; {career_catalog.university_name.nunique()} universities.')
print('Advisory baseline loaded. The uploaded outcome models are trained and activated in the extension below.')

Agent 6 catalog: 3,052 programs; 50 universities.
Advisory baseline loaded. The uploaded outcome models are trained and activated in the extension below.


## 2. Explainable fallback and sourced role examples

Research evidence includes thesis/dissertation/research/doctoral language. Industry
evidence includes internship, professional projects, deployment and applied design.
The fallback honors a stated goal only where program text supports it; otherwise it
shows the supported path or abstains. These are transparent heuristics, not a trained model.
Negation handling covers simple clauses; complex or mixed goals should be explicit.

Role titles are grounded in the U.S. Bureau of Labor Statistics occupational categories,
reviewed 2026-10-21. The program-to-role mapping is our advisory inference, not a BLS
endorsement or evidence of a particular university's graduate placements:
[Computing](https://www.bls.gov/ooh/computer-and-information-technology/home.htm),
[Engineering](https://www.bls.gov/ooh/architecture-and-engineering/home.htm),
[Math](https://www.bls.gov/ooh/math/home.htm),
[Business](https://www.bls.gov/ooh/business-and-financial/home.htm),
[Computing research](https://www.bls.gov/ooh/computer-and-information-technology/computer-and-information-research-scientists.htm).
Graduate-researcher suggestions outside computing are generic study directions from
program evidence, without an asserted occupational source or guaranteed position.

`avg_placement_stats` stays null unless a report matches the exact university/program,
country and dated cohort with valid surveyed/placed counts and source provenance.
The field holds that report's rate and definition, never an average of incompatible cohorts.

In [2]:
# Evidence rules are authored heuristics, never training labels or measured success probabilities.
RESEARCH_CUES = ('thesis', 'dissertation', 'research', 'doctoral', 'phd', 'experiments')
INDUSTRY_CUES = ('internship', 'industry', 'professional', 'capstone', 'deployment',
                 'manufacturing', 'software', 'design', 'management', 'engineering')
GOAL_RESEARCH = ('research', 'researcher', 'scientist', 'phd', 'doctoral', 'academia', 'professor')
GOAL_INDUSTRY = ('industry', 'job', 'employment', 'professional', 'developer', 'engineer', 'consulting', 'entrepreneur')

def positive_mentions(text, vocabulary):
    normalized = re.sub(r"[^a-z0-9\s.!?;,-]", ' ', career_text(text).casefold())
    found = []
    for phrase in vocabulary:
        for match in re.finditer(r'\b' + re.escape(phrase) + r'\b', normalized):
            # Negation in the same short clause suppresses the cue. This is conservative,
            # not a full language parser; ambiguous goals request explicit research/industry.
            clause = re.split(r'[.!?;,]|\b(?:but|however)\b', normalized[:match.start()])[-1]
            if not re.search(r'\b(?:no|not|never|without|avoid|neither)\b', ' '.join(clause.split()[-5:])):
                found.append(phrase)
                break
    return sorted(set(found))

def stated_career_goal(profile):
    explicit = career_key(profile.get('career_goal'))
    aliases = {'research': 'research', 'research-track': 'research', 'academia': 'research',
               'phd': 'research', 'industry': 'industry', 'job': 'industry', 'job-track': 'industry'}
    if explicit in aliases:
        return aliases[explicit], 'explicit career_goal'
    # Interests or academic merit are not silently treated as career intentions.
    text = career_text(profile.get('career_goal')) or career_text(profile.get('career_goals'))
    research, industry = positive_mentions(text, GOAL_RESEARCH), positive_mentions(text, GOAL_INDUSTRY)
    if bool(research) != bool(industry):
        return ('research' if research else 'industry'), 'interpreted free-text goal; confirm with student'
    return None, 'missing or mixed goal; set profile.career_goal to research or industry'

BLS_ENGINEERING = 'https://www.bls.gov/ooh/architecture-and-engineering/home.htm'
BLS_COMPUTING = 'https://www.bls.gov/ooh/computer-and-information-technology/home.htm'
BLS_MATH = 'https://www.bls.gov/ooh/math/home.htm'
BLS_BUSINESS = 'https://www.bls.gov/ooh/business-and-financial/home.htm'
# Program-to-occupation mappings are advisory inferences, not measured graduate destinations.
ROLE_FAMILIES = [
    ('cybersecurity', ('cybersecurity',), ['Information security analyst'], BLS_COMPUTING),
    ('data science', ('data science', 'artificial intelligence'), ['Data scientist', 'Software developer'], BLS_COMPUTING),
    ('statistics', ('statistics',), ['Statistician', 'Data scientist'], BLS_MATH),
    ('applied mathematics', ('applied mathematics',), ['Operations research analyst', 'Mathematician'], BLS_MATH),
    ('business analytics', ('business analytics',), ['Operations research analyst', 'Management analyst'], BLS_BUSINESS),
    ('computer engineering', ('computer engineering',), ['Computer hardware engineer'], BLS_ENGINEERING),
    ('electrical engineering', ('electrical engineering',), ['Electrical engineer', 'Electronics engineer'], BLS_ENGINEERING),
    ('biomedical engineering', ('biomedical engineering',), ['Biomedical engineer'], BLS_ENGINEERING),
    ('chemical engineering', ('chemical engineering',), ['Chemical engineer'], BLS_ENGINEERING),
    ('civil engineering', ('civil engineering',), ['Civil engineer'], BLS_ENGINEERING),
    ('industrial engineering', ('industrial engineering',), ['Industrial engineer'], BLS_ENGINEERING),
    ('mechanical engineering', ('mechanical engineering', 'robotics'), ['Mechanical engineer'], BLS_ENGINEERING),
    ('information systems', ('information systems',), ['Computer systems analyst'], BLS_COMPUTING),
    ('business administration', ('business administration',), ['Management analyst'], BLS_BUSINESS),
    ('computing', ('computer science', 'software engineering'), ['Software developer'], BLS_COMPUTING),
]

def suggested_roles(program_name, path):
    for family, phrases, industry_roles, source in ROLE_FAMILIES:
        if any(phrase in career_key(program_name) for phrase in phrases):
            if path == 'research':
                if family in {'computing', 'data science', 'cybersecurity', 'computer engineering', 'information systems'}:
                    return ['Computer and information research scientist'], [
                        'https://www.bls.gov/ooh/computer-and-information-technology/computer-and-information-research-scientists.htm']
                return ['Graduate researcher in ' + family], []
            if path == 'industry':
                return industry_roles, list(dict.fromkeys([source] + ([BLS_MATH] if family in {'data science', 'business analytics'} else [])))
    return [], []

PLACEMENT_COLUMNS = ['university_name', 'program_name', 'cohort_year', 'country', 'definition',
                     'placed_count', 'surveyed_count', 'source_url', 'verified_on']

def placement_for_program(reports, university, program, country, as_of):
    if reports is None or reports.empty:
        return None, 'No program-specific placement report supplied.'
    if set(PLACEMENT_COLUMNS) - set(reports):
        raise ValueError('Placement reports are missing required columns.')
    candidates = reports.loc[reports.university_name.map(career_key).eq(career_key(university))
        & reports.program_name.map(career_key).eq(career_key(program))].copy()
    if candidates.empty:
        return None, 'No report matches this exact university/program.'
    if candidates.duplicated(['university_name', 'program_name', 'cohort_year']).any():
        raise ValueError('Reconcile duplicate reports for the same program/cohort.')
    valid = []
    for row in candidates.to_dict('records'):
        try:
            placed, surveyed, year = float(row['placed_count']), float(row['surveyed_count']), float(row['cohort_year'])
            verified = pd.Timestamp(row['verified_on'])
        except (TypeError, ValueError):
            continue
        if (not all(math.isfinite(x) and x.is_integer() for x in (placed, surveyed, year))
                or not (0 <= placed <= surveyed and surveyed > 0 and 1900 <= year <= as_of.year)
                or pd.isna(verified) or verified.date() > as_of
                or not career_text(row['source_url']).startswith('https://')
                or not career_text(row['definition']) or not career_key(country)
                or career_key(row['country']) != career_key(country)):
            continue
        valid.append({**row, 'cohort_year': int(year), 'placed_count': int(placed), 'surveyed_count': int(surveyed),
                      'placement_rate': placed / surveyed})
    if not valid:
        return None, 'Matching report has invalid counts, date, country or provenance; statistics withheld.'
    latest = max(valid, key=lambda row: row['cohort_year'])
    return {key: latest[key] for key in PLACEMENT_COLUMNS + ['placement_rate']}, 'Source-specific reported cohort rate; not a forecast.'

def career_features(profile, program, catalog_row):
    # Identical feature construction is used in training and inference.
    goal = career_text(profile.get('career_goal')) or career_text(profile.get('career_goals'))
    return '\n'.join([career_text(program.get('program_name')), career_text(catalog_row.get('description_text')),
        career_text(catalog_row.get('curriculum_highlights')), career_text(catalog_row.get('faculty_research_areas')),
        career_text(profile.get('interests_text')), goal])

def career_agent(state, catalog=None, *, model_bundle=None, placement_reports=None, as_of=None):
    """Consume Agent 3 state['programs']; preserve its order and write state['careers']."""
    catalog = career_catalog if catalog is None else catalog
    if {'_university_key', '_program_key'} - set(catalog):
        raise ValueError('Pass a catalog prepared by load_career_catalog.')
    lookup = {(row['_university_key'], row['_program_key']): row for row in catalog.to_dict('records')}
    profile, rows = state.get('profile') or {}, []
    goal, goal_note = stated_career_goal(profile)
    as_of = pd.Timestamp.now(tz='Asia/Kolkata').date() if as_of is None else pd.Timestamp(as_of).date()
    if pd.isna(as_of):
        raise ValueError('as_of must be a valid date.')
    if model_bundle is not None and not model_bundle.get('independent_labels_confirmed'):
        raise ValueError('Model must come from verified independent outcome labels.')
    for program in state.get('programs') or []:
        university = career_text(program.get('university')) or career_text(program.get('university_name'))
        name = career_text(program.get('program_name'))
        found = lookup.get((career_key(university), career_key(name)))
        warning, research, industry = [], [], []
        path, alignment, confidence = None, None, None
        method = 'rule_based_advisory'
        if found is None:
            warning.append('Program absent from catalog; no track or placement inference made.')
        else:
            # Program body only; the student's declared goal cannot become its own evidence.
            body = ' '.join(career_text(found.get(key)) for key in ('description_text', 'curriculum_highlights', 'faculty_research_areas'))
            research, industry = positive_mentions(body, RESEARCH_CUES), positive_mentions(body, INDUSTRY_CUES)
            supports = {'research': len(research), 'industry': len(industry)}
            if model_bundle is not None:
                method = 'supervised_text_classifier'
                estimator = model_bundle['model']
                probabilities = estimator.predict_proba([career_features(profile, program, found)])[0]
                classes = list(estimator.classes_)
                confidence = float(max(probabilities))
                if confidence >= model_bundle['decision_threshold']:
                    path = str(classes[int(np.argmax(probabilities))])
                else:
                    warning.append('Supervised model confidence below the development threshold.')
                if goal is not None:
                    alignment = float(probabilities[classes.index(goal)])
                warning.append('Classifier scores are not calibrated placement probabilities.')
            else:
                options = [track for track, count in supports.items() if count]
                if goal in options:
                    path = goal
                elif len(options) == 1:
                    path = options[0]
                if goal is not None and sum(supports.values()):
                    alignment = supports[goal] / sum(supports.values())
                if path is None:
                    warning.append('Insufficient or mixed evidence; confirm the career goal and program pathway.')
                warning.append('Advisory rule result; no outcome-trained model is available.')
        if path is not None and goal is not None and path != goal:
            warning.append('Program pathway evidence conflicts with the stated goal; review this mismatch.')
        roles, sources = suggested_roles(name, path) if found is not None else ([], [])
        country = career_text(program.get('country')) or (career_text(found.get('country')) if found else '')
        placements, placement_note = placement_for_program(placement_reports, university, name, country, as_of)
        if found is None:
            placements = None
        rows.append({'university': university, 'program_name': name, 'country': country or None,
            'career_path': path, 'alignment_score': alignment, 'expected_roles': roles,
            'avg_placement_stats': placements, 'method': method,
            'alignment_score_type': ('uncalibrated_goal_class_probability' if model_bundle else 'heuristic_goal_evidence_share'),
            'model_confidence': confidence, 'stated_goal': goal, 'goal_interpretation': goal_note,
            'track_evidence': {'research': research, 'industry': industry},
            'role_sources': sources, 'roles_note': 'Illustrative roles to explore, not predicted job offers; qualifications may vary.',
            'placement_note': placement_note, 'review_required': path is None or goal is None or path != goal,
            'warnings': warning})
    return {**state, 'careers': rows, 'status': 'career_done'}

print('Active entry point: career_agent(state). Output key: careers. Missing admission scores are tolerated.')

Active entry point: career_agent(state). Output key: careers. Missing admission scores are tolerated.


## 3. Future supervised classifier: real outcomes only

The framework proposes an outcome-trained research/industry classifier. This implementation
provides a TF-IDF + logistic-regression baseline, not a fine-tuned transformer. Features use
program text, interests and the stated goal; labels must be independently observed outcomes
or independently adjudicated track decisions, **never copied from the goal or these rules**.
Freeze feature snapshots at advising time; do not include post-outcome evidence.

Create a train/dev CSV and a separate sealed test CSV with columns:
`record_id,student_id,split,university_name,program_name,interests_text,career_goal,outcome_track,label_source,observed_on`.
Labels are `research`/`industry`; maintain the same definition across splits (research roles
may occur in academia or companies). Assign student groups once; train/dev needs at least
30 students and both classes. Thirty students is only a minimum validation guard, not enough
to support a 96% confidence claim. No automatic training or test execution occurs.

```python
# Only after independently labelled data exists:
bundle = train_career_classifier("path/to/train_dev.csv", independent_labels_confirmed=True)
print(bundle["dev_report"])  # development only
freeze_career_classifier(bundle, "v1")
test_report = evaluate_career_test_once(bundle, "path/to/sealed_test.csv", "v1")
state = career_agent(state, model_bundle=bundle)
```

The primary test accuracy counts abstentions as errors; selective accuracy and coverage are
reported separately. Report all test rows. Repeated students prevent an independent-query
Wilson claim. After test inspection/tuning, use a fresh student cohort. Local locks are audit
guards, not access control. Never commit private outcomes or per-student notebook outputs.

In [3]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, balanced_accuracy_score, f1_score, confusion_matrix
from sklearn.pipeline import make_pipeline
from datetime import datetime, timezone
import pickle

OUTCOME_COLUMNS = ['record_id', 'student_id', 'split', 'university_name', 'program_name',
                   'interests_text', 'career_goal', 'outcome_track', 'label_source', 'observed_on']

def load_career_outcomes(path, allowed_splits):
    data = pd.read_csv(path, dtype=str, keep_default_na=False)
    if data.empty or set(OUTCOME_COLUMNS) - set(data):
        raise ValueError('Real career outcomes with all schema fields are required; no generated labels.')
    for column in OUTCOME_COLUMNS:
        data[column] = data[column].str.strip()
        if column not in ('interests_text', 'career_goal') and data[column].eq('').any():
            raise ValueError('Missing outcome field: ' + column)
    if not set(data.split).issubset(set(allowed_splits)):
        raise ValueError('File includes a forbidden split. Keep train/dev and test physically separate.')
    if (data.record_id.duplicated().any() or data.groupby('student_id').split.nunique().gt(1).any()
            or not data.outcome_track.isin(['research', 'industry']).all()):
        raise ValueError('Duplicate records, overlapping students or invalid outcome labels.')
    dates = pd.to_datetime(data.observed_on, errors='raise', utc=True)
    if (dates > pd.Timestamp.now(tz='UTC')).any():
        raise ValueError('Future outcomes cannot be used as ground truth.')
    lookup = {(r['_university_key'], r['_program_key']): r for r in career_catalog.to_dict('records')}
    texts = []
    for row in data.to_dict('records'):
        key = (career_key(row['university_name']), career_key(row['program_name']))
        if key not in lookup:
            raise ValueError('Outcome program absent from current catalog; no rows dropped.')
        texts.append(career_features(row, row, lookup[key]))
    data['feature_text'] = texts
    return data

def train_career_classifier(train_dev_path, *, independent_labels_confirmed=False, decision_threshold=.60):
    if not independent_labels_confirmed:
        raise ValueError('Confirm consent and independently sourced real outcomes, never goal-derived labels.')
    if not .5 <= decision_threshold <= 1:
        raise ValueError('decision_threshold must be between .5 and 1 and chosen using dev only.')
    data = load_career_outcomes(train_dev_path, {'train', 'dev'})
    if set(data.split) != {'train', 'dev'} or data.student_id.nunique() < 30:
        raise ValueError('Need both train and dev with at least 30 independent students in total.')
    train, dev = data.loc[data.split.eq('train')], data.loc[data.split.eq('dev')]
    if any(part.outcome_track.nunique() != 2 for part in (train, dev)):
        raise ValueError('Both outcome classes must be represented in train and dev.')
    train_text = set(train.feature_text.map(career_key))
    if train_text & set(dev.feature_text.map(career_key)):
        raise ValueError('Duplicate model inputs across train/dev; group them before splitting.')
    model = make_pipeline(TfidfVectorizer(ngram_range=(1, 2), min_df=1, sublinear_tf=True),
                          LogisticRegression(max_iter=2000, class_weight='balanced', random_state=42))
    model.fit(train.feature_text, train.outcome_track)
    prediction = model.predict(dev.feature_text)
    report = {'evaluation_type': 'development_only', 'students': int(dev.student_id.nunique()),
        'records': len(dev), 'accuracy': float(accuracy_score(dev.outcome_track, prediction)),
        'balanced_accuracy': float(balanced_accuracy_score(dev.outcome_track, prediction)),
        'macro_f1': float(f1_score(dev.outcome_track, prediction, average='macro'))}
    return {'model': model, 'independent_labels_confirmed': True, 'decision_threshold': decision_threshold,
        'training_students': sorted(set(data.student_id)),
        'training_input_hashes': sorted({hashlib.sha256(career_key(text).encode()).hexdigest() for text in data.feature_text}),
        'training_data_sha256': hashlib.sha256(Path(train_dev_path).read_bytes()).hexdigest(), 'dev_report': report}

def career_model_fingerprint(bundle):
    # Both executable inference code and the fitted classifier/configuration are frozen.
    import marshal, types
    def stable_code(item):
        constants = tuple(stable_code(value) if isinstance(value, types.CodeType) else value for value in item.co_consts)
        return item.replace(co_filename='<career-evaluation>', co_firstlineno=1, co_consts=constants)
    digest = hashlib.sha256(pickle.dumps(bundle, protocol=5))
    for function in (career_agent, career_features, stated_career_goal, career_text, career_key, positive_mentions):
        digest.update(marshal.dumps(stable_code(function.__code__)))
    digest.update(json.dumps([RESEARCH_CUES, INDUSTRY_CUES, GOAL_RESEARCH, GOAL_INDUSTRY]).encode())
    digest.update(career_catalog.to_json(orient='records').encode())
    return digest.hexdigest()

def freeze_career_classifier(bundle, version):
    if not re.fullmatch(r'[A-Za-z0-9_-]+', version) or not bundle.get('independent_labels_confirmed'):
        raise ValueError('Use a unique simple version name and a verified outcome-trained bundle.')
    AGENT6_RUNS.mkdir(parents=True, exist_ok=True)
    with (AGENT6_RUNS / f'{version}.freeze.json').open('x', encoding='utf-8') as handle:
        json.dump({'version': version, 'fingerprint': career_model_fingerprint(bundle),
                   'frozen_at': datetime.now(timezone.utc).isoformat()}, handle, indent=2)

def evaluate_career_test_once(bundle, test_path, version):
    freeze = json.loads((AGENT6_RUNS / f'{version}.freeze.json').read_text(encoding='utf-8'))
    if freeze['fingerprint'] != career_model_fingerprint(bundle):
        raise ValueError('Classifier/catalog changed after freezing.')
    test = load_career_outcomes(test_path, {'test'})
    if set(test.student_id) & set(bundle['training_students']):
        raise ValueError('Test students overlap train/dev.')
    test_hashes = {hashlib.sha256(career_key(text).encode()).hexdigest() for text in test.feature_text}
    if test_hashes & set(bundle['training_input_hashes']):
        raise ValueError('Test model inputs duplicate train/dev inputs.')
    for path in AGENT6_RUNS.glob('*.consumed.json'):
        if set(test.student_id) & set(json.loads(path.read_text(encoding='utf-8'))['student_ids']):
            raise ValueError('Test cohort already consumed; collect fresh students after tuning.')
    with (AGENT6_RUNS / f'{version}.consumed.json').open('x', encoding='utf-8') as handle:
        json.dump({'student_ids': sorted(set(test.student_id)), 'test_sha256': hashlib.sha256(Path(test_path).read_bytes()).hexdigest()}, handle)
    prediction, confidence = [], []
    # Test the same public career_agent used by the pipeline, including abstention.
    for row in test.to_dict('records'):
        state = {'profile': row, 'programs': [{'university': row['university_name'], 'program_name': row['program_name']}]}
        result = career_agent(state, model_bundle=bundle)['careers'][0]
        prediction.append(result['career_path'] or 'unknown')
        confidence.append(result['model_confidence'])
    predicted = np.array(prediction)
    decided = predicted != 'unknown'
    correct = predicted == test.outcome_track.to_numpy()
    count = int(decided.sum())
    overall = float(correct.mean())  # abstentions count as incorrect in this primary metric
    z, n = 1.959963984540054, len(test)
    centre = (overall + z*z/(2*n))/(1+z*z/n)
    radius = z*math.sqrt(overall*(1-overall)/n + z*z/(4*n*n))/(1+z*z/n)
    independent = test.student_id.nunique() == n
    report = {'version': version, 'fingerprint': freeze['fingerprint'], 'test_records': n,
        'test_students': int(test.student_id.nunique()), 'accuracy_including_abstentions': overall,
        'decision_coverage': float(decided.mean()), 'unknown_share': float((~decided).mean()),
        'selective_accuracy': float(correct[decided].mean()) if count else None,
        'accuracy_wilson_95': [max(0, centre-radius), min(1, centre+radius)],
        'wilson_independent_students': bool(independent),
        'target_96_supported': bool(independent and centre-radius >= .96),
        'confusion_matrix': pd.crosstab(test.outcome_track, pd.Series(prediction, index=test.index)).to_dict()}
    with (AGENT6_RUNS / f'{version}.report.json').open('x', encoding='utf-8') as handle:
        json.dump(report, handle, indent=2)
    test.assign(prediction=prediction, confidence=confidence).to_json(
        AGENT6_RUNS / f'{version}.predictions.json', orient='records', indent=2)
    return report

career_model_bundle = None
print('Supervised training is available explicitly, but no model has been trained on invented labels.')

Supervised training is available explicitly, but no model has been trained on invented labels.


## 4. Execute the actual Agent 3 -> Agent 6 handoff

The following demonstration calls the real `program_agent` and then `career_agent` with
real catalog entries. Its profile and shortlist are illustrative, not real student test data.
The full-catalog run measures processing coverage only, not career-outcome accuracy.

In [4]:
# Import only Agent 3's actual setup/implementation cell into an isolated namespace.
# This does not read its judgments, run its diagnostics, or consume any frozen test cohort.
import contextlib
import io
agent3_path = PROJECT_ROOT / 'backend/app/agents/program_agent/Agent3_Program_agent.ipynb'
agent3_notebook = json.loads(agent3_path.read_text(encoding='utf-8'))
setup = next(''.join(c['source']) for c in agent3_notebook['cells']
             if c['cell_type'] == 'code' and 'class ProgramRankerV2:' in ''.join(c['source']))
agent3_kernel = {}
with contextlib.redirect_stdout(io.StringIO()):
    exec(compile(setup, 'Agent3_implementation_for_career_handoff', 'exec'), agent3_kernel)

demo_profile = {'interests_text': 'distributed systems, cloud infrastructure and backend engineering',
    'target_program': 'Computer Science', 'target_degree': 'MS', 'career_goal': 'industry',
    'gpa': 8.6, 'gpa_scale': 10, 'profile_strength_score': 72}
demo_shortlist = {'profile': demo_profile, 'matched_universities': [
    {'university': 'Stanford University', 'country': 'United States', 'programs': [
        {'program_name': 'MS Computer Science', 'match_score': .787, 'admit_probability': .566, 'tier': 'Target'},
        {'program_name': 'MS Computer Science - Systems', 'match_score': .793, 'admit_probability': .566, 'tier': 'Target'}]}]}
agent3_live_demo = agent3_kernel['program_agent'](demo_shortlist)
demo_original = deepcopy(agent3_live_demo)
agent6_live_demo = career_agent(agent3_live_demo)
assert agent3_live_demo == demo_original
assert [(r['university'], r['program_name']) for r in agent6_live_demo['careers']] == [
    (r['university'], r['program_name']) for r in agent3_live_demo['programs']]
assert all(r['avg_placement_stats'] is None for r in agent6_live_demo['careers'])
assert agent6_live_demo['careers'][0]['program_name'] == 'MS Computer Science - Systems'
print('REAL Agent 3 -> Agent 6 function handoff; illustrative student and shortlist, real program catalog:')
print(pd.DataFrame(agent6_live_demo['careers'])[['university', 'program_name', 'career_path',
    'alignment_score', 'expected_roles', 'method']].to_string(index=False))
print('Placement statistics: unknown (no source report supplied). Alignment scores are heuristic evidence shares.')

# Coverage check across all real program identities; no career-outcome labels are inferred.
coverage_state = {'profile': {'career_goal': 'industry'}, 'programs': [
    {'university': r['university_name'], 'program_name': r['program_name'], 'country': r.get('country')}
    for r in career_catalog.to_dict('records')]}
career_catalog_result = career_agent(coverage_state)
career_catalog_counts = {'input_programs': len(coverage_state['programs']),
    'output_programs': len(career_catalog_result['careers']),
    'track_decisions': sum(r['career_path'] is not None for r in career_catalog_result['careers']),
    'with_role_examples': sum(bool(r['expected_roles']) for r in career_catalog_result['careers']),
    'with_verified_placement_stats': sum(r['avg_placement_stats'] is not None for r in career_catalog_result['careers'])}
assert career_catalog_counts['input_programs'] == career_catalog_counts['output_programs']
print('Catalog execution coverage (not accuracy):', career_catalog_counts)

REAL Agent 3 -> Agent 6 function handoff; illustrative student and shortlist, real program catalog:
         university                  program_name career_path  alignment_score       expected_roles              method
Stanford University MS Computer Science - Systems    industry         0.666667 [Software developer] rule_based_advisory
Stanford University           MS Computer Science    industry         0.666667 [Software developer] rule_based_advisory
Placement statistics: unknown (no source report supplied). Alignment scores are heuristic evidence shares.
Catalog execution coverage (not accuracy): {'input_programs': 3052, 'output_programs': 3052, 'track_decisions': 3052, 'with_role_examples': 3052, 'with_verified_placement_stats': 0}


## 5. Executed functional prediction checks — explicitly fictional

In [5]:
# Fictional examples independently specify expected rule behavior, not real career labels.
fixture_career_catalog = pd.DataFrame([
    {'university_name': 'Fixture University', 'program_name': 'Research Computing',
     'description_text': 'Thesis and dissertation research with experiments.', 'curriculum_highlights': '', 'faculty_research_areas': '', 'country': 'United States'},
    {'university_name': 'Fixture University', 'program_name': 'Professional Computing',
     'description_text': 'Professional software design, industry internship and capstone.', 'curriculum_highlights': '', 'faculty_research_areas': '', 'country': 'United States'},
    {'university_name': 'Fixture University', 'program_name': 'Mixed Computing',
     'description_text': 'Research thesis plus industry internship and software design.', 'curriculum_highlights': '', 'faculty_research_areas': '', 'country': 'United States'},
    {'university_name': 'Fixture University', 'program_name': 'No Evidence',
     'description_text': '', 'curriculum_highlights': '', 'faculty_research_areas': '', 'country': 'United States'},
    {'university_name': 'Fixture University', 'program_name': 'Negated Research',
     'description_text': 'No thesis. No research. Industry internship and professional capstone.', 'curriculum_highlights': '', 'faculty_research_areas': '', 'country': 'United States'},
])
fixture_career_catalog['_university_key'] = fixture_career_catalog.university_name.map(career_key)
fixture_career_catalog['_program_key'] = fixture_career_catalog.program_name.map(career_key)
career_cases = [
    ('Research goal and research evidence', 'Research Computing', {'career_goal': 'research'}, 'research'),
    ('Industry goal and professional evidence', 'Professional Computing', {'career_goal': 'industry'}, 'industry'),
    ('Mixed program honors research goal', 'Mixed Computing', {'career_goal': 'research'}, 'research'),
    ('Mixed program honors industry goal', 'Mixed Computing', {'career_goal': 'industry'}, 'industry'),
    ('Missing goal, mixed program abstains', 'Mixed Computing', {}, None),
    ('No descriptive evidence abstains', 'No Evidence', {'career_goal': 'research'}, None),
    ('Unknown program abstains', 'Absent Program', {'career_goal': 'industry'}, None),
    ('Opposite-only evidence flags mismatch', 'Professional Computing', {'career_goal': 'research'}, 'industry'),
    ('Negated research is not positive evidence', 'Negated Research', {'career_goal': 'industry'}, 'industry'),
    ('Negated free-text research goal', 'Mixed Computing', {'career_goals': 'I do not want research; I want an industry job.'}, 'industry'),
    ('Mixed free-text goal abstains', 'Mixed Computing', {'career_goals': 'Research and industry both interest me'}, None),
    ('Interests alone do not invent a goal', 'Mixed Computing', {'interests_text': 'research scientist'}, None),
]
career_check_rows = []
for name, program, profile, expected in career_cases:
    state = {'profile': profile, 'programs': [{'university': 'Fixture University', 'program_name': program,
                                              'admit_probability': None, 'tier': 'Unknown'}]}
    original = deepcopy(state)
    row = career_agent(state, fixture_career_catalog)['careers'][0]
    passed = row['career_path'] == expected and original == state and row['avg_placement_stats'] is None
    if name == 'Opposite-only evidence flags mismatch':
        passed = passed and row['alignment_score'] == 0
    career_check_rows.append({'case': name, 'expected': expected, 'predicted': row['career_path'], 'passed': passed})
assert career_agent({'profile': {}})['careers'] == []
career_check_rows.append({'case': 'Empty Agent 3 shortlist', 'expected': 'empty', 'predicted': 'empty', 'passed': True})

# A source-backed rate is taken from one exact cohort, never from unrelated job statistics.
placement_fixture = pd.DataFrame([{'university_name': 'Fixture University', 'program_name': 'Professional Computing',
    'cohort_year': 2025, 'country': 'United States', 'definition': 'Fictional fixture: employed within six months among respondents',
    'placed_count': 80, 'surveyed_count': 100, 'source_url': 'https://example.org/fictional-fixture', 'verified_on': '2026-01-01'}])
fixture_program_state = {'profile': {'career_goal': 'industry'}, 'programs': [
    {'university': 'Fixture University', 'program_name': 'Professional Computing'}]}
row = career_agent(fixture_program_state, fixture_career_catalog, placement_reports=placement_fixture, as_of='2026-10-21')['careers'][0]
career_check_rows.append({'case': 'Explicit cohort counts compute rate', 'expected': .8,
                         'predicted': row['avg_placement_stats']['placement_rate'], 'passed': row['avg_placement_stats']['placement_rate'] == .8})
invalid_report = placement_fixture.copy()
invalid_report['placed_count'] = 120
row = career_agent(fixture_program_state, fixture_career_catalog, placement_reports=invalid_report, as_of='2026-10-21')['careers'][0]
career_check_rows.append({'case': 'Impossible placement counts withheld', 'expected': None,
                         'predicted': row['avg_placement_stats'], 'passed': row['avg_placement_stats'] is None})
career_functional_checks = pd.DataFrame(career_check_rows)
print(career_functional_checks.to_string(index=False))
assert career_functional_checks.passed.all(), 'A displayed career contract case failed.'
print('Functional checks only; these fixture percentages are not outcome-classifier accuracy.')

                                     case expected predicted  passed
      Research goal and research evidence research  research    True
  Industry goal and professional evidence industry  industry    True
       Mixed program honors research goal research  research    True
       Mixed program honors industry goal industry  industry    True
     Missing goal, mixed program abstains     None      None    True
         No descriptive evidence abstains     None      None    True
                 Unknown program abstains     None      None    True
    Opposite-only evidence flags mismatch industry  industry    True
Negated research is not positive evidence industry  industry    True
          Negated free-text research goal industry  industry    True
            Mixed free-text goal abstains     None      None    True
     Interests alone do not invent a goal     None      None    True
                  Empty Agent 3 shortlist    empty     empty    True
      Explicit cohort counts compu

## 6. Use in the existing pipeline

```python
state = program_agent(state)
state = career_agent(state)  # adds careers; retains programs, finance and scholarships
```

Future placement report schema: `university_name,program_name,cohort_year,country,definition,placed_count,surveyed_count,source_url,verified_on`.
Load a reviewed DataFrame and pass `placement_reports=reports`. URLs and dates record human
provenance; their presence alone cannot prove that a report was verified.

## Historical advisory baseline results — final enriched results follow below

In [6]:
print('AGENT 6 - LEGACY ADVISORY BASELINE RESULTS')
print('Method: rule-based advisory fallback; no independent student track labels supplied.')
print(f'Functional prediction checks: {int(career_functional_checks.passed.sum())}/{len(career_functional_checks)} '
      f'passed ({career_functional_checks.passed.mean():.2%}). This is a check pass rate, not predictive accuracy.')
print('Agent 3 -> Agent 6 handoff: PASSED; Agent 3 order and caller state preserved.')
print('Real-catalog coverage:', career_catalog_counts)
saved_career_reports = sorted(AGENT6_RUNS.glob('*.report.json'), key=lambda p: p.stat().st_mtime)
if saved_career_reports:
    saved_career_test = json.loads(saved_career_reports[-1].read_text(encoding='utf-8'))
    print('Saved supervised test report (not rerun; separate from the active rule fallback):', saved_career_test)
else:
    print('Independent research/industry classification accuracy: NOT MEASURED.')
    print('96% real-world accuracy claim: NOT VERIFIED.')
print('Placement statistics: NOT AVAILABLE without source-backed program/cohort reports.')
print('Example career predictions:')
print(pd.DataFrame(agent6_live_demo['careers'])[['program_name', 'career_path', 'expected_roles', 'alignment_score']].to_string(index=False))

AGENT 6 - LEGACY ADVISORY BASELINE RESULTS
Method: rule-based advisory fallback; no independent student track labels supplied.
Functional prediction checks: 15/15 passed (100.00%). This is a check pass rate, not predictive accuracy.
Agent 3 -> Agent 6 handoff: PASSED; Agent 3 order and caller state preserved.
Real-catalog coverage: {'input_programs': 3052, 'output_programs': 3052, 'track_decisions': 3052, 'with_role_examples': 3052, 'with_verified_placement_stats': 0}
Independent research/industry classification accuracy: NOT MEASURED.
96% real-world accuracy claim: NOT VERIFIED.
Placement statistics: NOT AVAILABLE without source-backed program/cohort reports.
Example career predictions:
                 program_name career_path       expected_roles  alignment_score
MS Computer Science - Systems    industry [Software developer]         0.666667
          MS Computer Science    industry [Software developer]         0.666667


## Dataset audit and institution-separated splits

In [7]:
import joblib
import sklearn
from sklearn.model_selection import GroupShuffleSplit
from sklearn.preprocessing import OrdinalEncoder, FunctionTransformer, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.compose import TransformedTargetRegressor
from sklearn.cluster import KMeans
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score, silhouette_score, davies_bouldin_score
from threadpoolctl import threadpool_limits
from datetime import datetime, timezone

CAREER_DATA_PATH = AGENT6_DATA / 'agent_6_dataset_final_merged.csv'
CAREER_MODEL_DIR = PROJECT_ROOT / 'backend/app/agents/career_agent/trained_models'
CAREER_REPORT_DIR = PROJECT_ROOT / 'data/processed/agent6'
CAREER_MODEL_DIR.mkdir(parents=True, exist_ok=True)
CAREER_REPORT_DIR.mkdir(parents=True, exist_ok=True)
outcomes = pd.read_csv(CAREER_DATA_PATH, low_memory=False)
data_fingerprint = hashlib.sha256(CAREER_DATA_PATH.read_bytes()).hexdigest()
REG_FEATURES = ['cip_family','cip_code','credential_level','control']
CLUSTER_FEATURES = ['earnings_1yr','earnings_4yr','median_debt_observed']
observed = outcomes.loc[outcomes.row_type.eq('scorecard_observed') & outcomes.in_us_scope.eq(True) & outcomes.unitid.notna()].copy()
identity = ['unitid','cip_code','credential_level']
duplicates = int(observed.duplicated(identity).sum())
# Select evidence count, never the target value, to resolve duplicate field/cohort keys.
observed = observed.sort_values('n_earnings',ascending=False,kind='stable').drop_duplicates(identity).reset_index(drop=True)
for c in CLUSTER_FEATURES:
    observed[c] = pd.to_numeric(observed[c],errors='coerce')
    observed.loc[observed[c] < 0,c] = np.nan
splitter = GroupShuffleSplit(n_splits=1,test_size=.20,random_state=42)
development_i,test_i = next(splitter.split(observed,groups=observed.unitid))
train_rel,dev_rel = next(GroupShuffleSplit(n_splits=1,test_size=.25,random_state=43).split(
    observed.iloc[development_i],groups=observed.iloc[development_i].unitid))
train_i,dev_i = development_i[train_rel],development_i[dev_rel]
split_names = pd.Series('',index=observed.index)
for name,indices in [('train',train_i),('dev',dev_i),('test',test_i)]: split_names.loc[indices]=name
assert not (set(observed.iloc[train_i].unitid)&set(observed.iloc[dev_i].unitid))
assert not (set(observed.iloc[development_i].unitid)&set(observed.iloc[test_i].unitid))
audit6 = {'uploaded_rows':len(outcomes),'uploaded_columns':len(outcomes.columns),'sha256':data_fingerprint,
    'row_types':outcomes.row_type.value_counts().to_dict(),'deduplicated_us_observed_rows':len(observed),
    'duplicate_observed_keys_removed':duplicates,'mapped_program_ids':int(outcomes.program_id.nunique()),
    'split_rows':split_names.value_counts().to_dict(),
    'split_institutions':{n:int(observed.loc[split_names.eq(n)].unitid.nunique()) for n in ['train','dev','test']},
    'split_institution_overlap':0,'research_industry_ground_truth_labels':0,
    'placement_status_counts':outcomes.placement_data_status.value_counts().to_dict(),
    'provenance':'Source names and scopes are recorded in the uploaded file; row-level source reconstruction and cohort-year validation are not available.',
    'population_scope':'Uploaded file marks outcomes as proxies for US federal-aid completers, not international-student forecasts.'}
observed[identity].assign(split=split_names).to_csv(CAREER_REPORT_DIR/'institution_splits.csv',index=False)
print(json.dumps(audit6,indent=2))
print('No heuristic job/research score, admission label, or employment proxy is used as a supervised target.')



{
  "uploaded_rows": 51842,
  "uploaded_columns": 97,
  "sha256": "74c730ed9566c876a748155ac0d6a46e91139194382f28aad6d04136e354d506",
  "row_types": {
    "scorecard_observed": 48809,
    "program_estimated": 3033
  },
  "deduplicated_us_observed_rows": 48679,
  "duplicate_observed_keys_removed": 0,
  "mapped_program_ids": 3033,
  "split_rows": {
    "train": 29276,
    "dev": 10208,
    "test": 9195
  },
  "split_institutions": {
    "train": 1371,
    "dev": 457,
    "test": 457
  },
  "split_institution_overlap": 0,
  "research_industry_ground_truth_labels": 0,
  "placement_status_counts": {
    "proxy_only": 51789,
    "real_counts": 42,
    "real_rate_only": 11
  },
  "provenance": "Source names and scopes are recorded in the uploaded file; row-level source reconstruction and cohort-year validation are not available.",
  "population_scope": "Uploaded file marks outcomes as proxies for US federal-aid completers, not international-student forecasts."
}
No heuristic job/research scor

## Supervised four-year cohort earnings regression

In [8]:
def regression_inputs(frame):
    return frame[REG_FEATURES].fillna('Unknown').astype(str)


def earnings_metrics(actual,predicted):
    actual,predicted=np.asarray(actual),np.asarray(predicted)
    return {'n':len(actual),'mae_usd':float(mean_absolute_error(actual,predicted)),
            'rmse_usd':float(np.sqrt(mean_squared_error(actual,predicted))),
            'r2':float(r2_score(actual,predicted)),
            'within_10_percent':float(np.mean(np.abs(predicted-actual)/actual <= .10)),
            'median_absolute_percentage_error':float(np.median(np.abs(predicted-actual)/actual))}


def make_earnings_model():
    # Rare category grouping is fitted on training institutions only. No university identity,
    # earnings-derived field aggregates, debt/earnings ratios, or placement proxies enter X.
    return Pipeline([
        ('categories',OrdinalEncoder(handle_unknown='use_encoded_value',unknown_value=-1,
                                     encoded_missing_value=-1,max_categories=200)),
        ('regressor',TransformedTargetRegressor(
            regressor=HistGradientBoostingRegressor(categorical_features=[0,1,2,3],
                max_iter=150,max_leaf_nodes=15,min_samples_leaf=30,l2_regularization=5,
                early_stopping=False,random_state=42),func=np.log1p,inverse_func=np.expm1))])


eligible_earnings = observed.earnings_4yr.notna() & observed.earnings_4yr.gt(0)
reg_train=observed.loc[split_names.eq('train') & eligible_earnings]
reg_dev=observed.loc[split_names.eq('dev') & eligible_earnings]
reg_test=observed.loc[split_names.eq('test') & eligible_earnings]
earnings_model=make_earnings_model()
with threadpool_limits(limits=1):
    earnings_model.fit(regression_inputs(reg_train),reg_train.earnings_4yr)
    dev_regression=earnings_metrics(reg_dev.earnings_4yr,earnings_model.predict(regression_inputs(reg_dev)))
print('Development results (not test accuracy):',dev_regression)
# Fixed hyperparameters; refit on train+dev, freeze, then evaluate test once below.
reg_fit=observed.loc[~split_names.eq('test') & eligible_earnings]
with threadpool_limits(limits=1):
    earnings_model.fit(regression_inputs(reg_fit),reg_fit.earnings_4yr)
joblib.dump(earnings_model,CAREER_MODEL_DIR/'agent6_earnings_model.joblib',compress=3)
regression_baseline=float(reg_fit.earnings_4yr.median())
print('Saved fitted earnings model; withheld test institutions have not been used for fitting.')



Development results (not test accuracy): {'n': 8036, 'mae_usd': 9570.528011246099, 'rmse_usd': 14735.56873678429, 'r2': 0.6675033025692223, 'within_10_percent': 0.49800895968143355, 'median_absolute_percentage_error': 0.10042989757975354}
Saved fitted earnings model; withheld test institutions have not been used for fitting.


## Unsupervised outcome clustering, selected on development institutions

In [9]:
eligible_clusters=observed[CLUSTER_FEATURES].notna().sum(axis=1).ge(2)
cluster_train=observed.loc[split_names.eq('train') & eligible_clusters,CLUSTER_FEATURES]
cluster_dev=observed.loc[split_names.eq('dev') & eligible_clusters,CLUSTER_FEATURES]
cluster_fit=observed.loc[~split_names.eq('test') & eligible_clusters,CLUSTER_FEATURES]
cluster_test=observed.loc[split_names.eq('test') & eligible_clusters,CLUSTER_FEATURES]


def cluster_preprocessor():
    return Pipeline([('log',FunctionTransformer(np.log1p,feature_names_out='one-to-one')),
                     ('imputer',SimpleImputer(strategy='median')),
                     ('scale',StandardScaler())])


pre=cluster_preprocessor()
with threadpool_limits(limits=1):
    train_z=pre.fit_transform(cluster_train);dev_z=pre.transform(cluster_dev)
    cluster_candidates=[]
    for k in [2,3,4,5,6,7,8]:
        candidate=KMeans(n_clusters=k,n_init=10,random_state=42).fit(train_z)
        labels=candidate.predict(dev_z)
        score=float(silhouette_score(dev_z,labels,sample_size=min(2000,len(dev_z)),random_state=42))
        cluster_candidates.append({'k':k,'development_silhouette':score})
    best_k=max(cluster_candidates,key=lambda row:row['development_silhouette'])['k']
    cluster_model=Pipeline([('preprocess',cluster_preprocessor()),
                            ('kmeans',KMeans(n_clusters=best_k,n_init=10,random_state=42))])
    cluster_model.fit(cluster_fit)
joblib.dump(cluster_model,CAREER_MODEL_DIR/'agent6_outcome_clusters.joblib',compress=3)
cluster_fit_summary=cluster_fit.copy()
with threadpool_limits(limits=1): cluster_fit_summary['cluster']=cluster_model.predict(cluster_fit)
cluster_summary=cluster_fit_summary.groupby('cluster').agg(
    count=('earnings_4yr','size'),median_earnings_1yr=('earnings_1yr','median'),
    median_earnings_4yr=('earnings_4yr','median'),median_observed_debt=('median_debt_observed','median'))
cluster_summary.to_csv(CAREER_REPORT_DIR/'cluster_profiles.csv')
print(pd.DataFrame(cluster_candidates).to_string(index=False))
print('Selected k:',best_k,'(development selection only)')
print(cluster_summary.to_string())
print('Clusters describe earnings/debt patterns. They are not research/industry labels or job-placement classes.')



 k  development_silhouette
 2                0.420787
 3                0.415779
 4                0.308738
 5                0.330069
 6                0.323279
 7                0.277787
 8                0.271873
Selected k: 2 (development selection only)
         count  median_earnings_1yr  median_earnings_4yr  median_observed_debt
cluster                                                                       
0        11537              74913.0              89820.0               28106.0
1        17906              42967.0              55396.0               21447.0
Clusters describe earnings/debt patterns. They are not research/industry labels or job-placement classes.


## Frozen evaluation on withheld institutions and portable joblib bundle

In [10]:
frozen_hashes={name:hashlib.sha256((CAREER_MODEL_DIR/name).read_bytes()).hexdigest()
              for name in ['agent6_earnings_model.joblib','agent6_outcome_clusters.joblib']}
with threadpool_limits(limits=1):
    test_earnings_predictions=earnings_model.predict(regression_inputs(reg_test))
    test_cluster_labels=cluster_model.predict(cluster_test)
    test_z=cluster_model.named_steps['preprocess'].transform(cluster_test)
    cluster_test_report={'silhouette':float(silhouette_score(test_z,test_cluster_labels,
        sample_size=min(2000,len(test_z)),random_state=42)),
        'davies_bouldin':float(davies_bouldin_score(test_z,test_cluster_labels)),
        'rows':len(cluster_test),'cluster_count':best_k}
regression_test_report=earnings_metrics(reg_test.earnings_4yr,test_earnings_predictions)
baseline_test_report=earnings_metrics(reg_test.earnings_4yr,np.full(len(reg_test),regression_baseline))
reg_test[identity+['earnings_4yr']].assign(predicted_earnings_4yr=test_earnings_predictions).to_csv(
    CAREER_REPORT_DIR/'earnings_test_predictions.csv',index=False)
print('Held-out earnings model:',regression_test_report)
print('Training-median baseline:',baseline_test_report)
print('Held-out clustering quality (not accuracy):',cluster_test_report)

# Map to exact Agent 2 catalog IDs. Never join on a university alone.
mapped_outcomes=outcomes.loc[outcomes.program_id.notna()].copy()
assert mapped_outcomes.program_id.is_unique
catalog_ids=pd.read_csv(AGENT6_CATALOG_PATH)
assert catalog_ids.program_id.is_unique
mapped_outcomes=mapped_outcomes.merge(catalog_ids[['program_id','unitid','university_name','program_name','degree_type','institution_type']],
    on='program_id',suffixes=('','_catalog'),how='inner',validate='one_to_one')
assert (mapped_outcomes.unitid.isna() | mapped_outcomes.unitid.eq(mapped_outcomes.unitid_catalog)).all()
assert (mapped_outcomes.program_label_agent2 == mapped_outcomes.program_name_catalog).all()
mapped_outcomes['outcome_source_unitid']=mapped_outcomes.unitid
mapped_outcomes['unitid_source']=np.where(mapped_outcomes.unitid.isna(),'exact_catalog_program_identity','uploaded_outcome_identity')
mapped_outcomes['unitid']=mapped_outcomes.unitid.fillna(mapped_outcomes.unitid_catalog)
mapped_outcomes['control']=mapped_outcomes.control.fillna(mapped_outcomes.institution_type.str.lower())
assert mapped_outcomes.credential_level.eq("Master's").all()
assert mapped_outcomes.degree_type.isin(['MS','MEng','MBA']).all()
with threadpool_limits(limits=1):
    mapped_outcomes['model_earnings_4yr']=earnings_model.predict(regression_inputs(mapped_outcomes))
    usable=mapped_outcomes[CLUSTER_FEATURES].notna().sum(axis=1).ge(2)
    mapped_outcomes['outcome_cluster']=pd.Series(pd.NA,index=mapped_outcomes.index,dtype='Int64')
    mapped_outcomes.loc[usable,'outcome_cluster']=cluster_model.predict(mapped_outcomes.loc[usable,CLUSTER_FEATURES])
lookup_bundle={
    'version':'agent6-outcomes-v2','dataset_sha256':data_fingerprint,
    'regression_features':REG_FEATURES,'cluster_features':CLUSTER_FEATURES,
    'programs':json.loads(mapped_outcomes.to_json(orient='records')),
    'cluster_profiles':json.loads(cluster_summary.reset_index().to_json(orient='records')),
    'regression_test':regression_test_report,'baseline_test':baseline_test_report,
    'clustering_test':cluster_test_report,'sklearn_version':sklearn.__version__,
    'scope':'Uploaded cohort observations/estimates, not individual career outcomes; no international salary forecast.'}
joblib.dump(lookup_bundle,CAREER_MODEL_DIR/'agent6_lookup_bundle.joblib',compress=3)
for name,digest in frozen_hashes.items():
    assert hashlib.sha256((CAREER_MODEL_DIR/name).read_bytes()).hexdigest()==digest
print('Saved 3 portable joblib files. Only built-in sklearn estimators and plain-data dictionaries are serialized.')



Held-out earnings model: {'n': 7171, 'mae_usd': 10321.838202915358, 'rmse_usd': 16067.05852850458, 'r2': 0.6342611833871428, 'within_10_percent': 0.48180170129689026, 'median_absolute_percentage_error': 0.10457218342811397}
Training-median baseline: {'n': 7171, 'mae_usd': 19150.852740203598, 'rmse_usd': 27460.870617125034, 'r2': -0.0683828496048291, 'within_10_percent': 0.2302328824431739, 'median_absolute_percentage_error': 0.22035343263223275}
Held-out clustering quality (not accuracy): {'silhouette': 0.41344629396972776, 'davies_bouldin': 1.0153422526931908, 'rows': 6672, 'cluster_count': 2}
Saved 3 portable joblib files. Only built-in sklearn estimators and plain-data dictionaries are serialized.


## Active career agent: exact program evidence, clustering and conservative placement scope

In [11]:
career_agent_legacy=career_agent


def load_agent6_models(directory=CAREER_MODEL_DIR):
    directory=Path(directory)
    return {'earnings':joblib.load(directory/'agent6_earnings_model.joblib'),
            'clusters':joblib.load(directory/'agent6_outcome_clusters.joblib'),
            'lookup':joblib.load(directory/'agent6_lookup_bundle.joblib')}


agent6_models=load_agent6_models()


def career_agent_v2(state,catalog=None,*,model_bundle=None,placement_reports=None,as_of=None,artifacts=None):
    artifacts=agent6_models if artifacts is None else artifacts
    # Preserve existing documented career-path rules and optional independently labelled classifier.
    result=career_agent_legacy(state,catalog=catalog,model_bundle=model_bundle,
                               placement_reports=placement_reports,as_of=as_of)
    reference_year=(pd.Timestamp.now(tz='Asia/Kolkata') if as_of is None else pd.Timestamp(as_of)).year
    entries=artifacts['lookup']['programs']
    by_id={r['program_id']:r for r in entries}
    by_name={(career_key(r['university_name']),career_key(r['program_name_catalog'])):r for r in entries}
    for source,row in zip(state.get('programs') or [],result['careers']):
        key=(career_key(row['university']),career_key(row['program_name']))
        evidence=by_id.get(source.get('program_id')) if source.get('program_id') else by_name.get(key)
        if evidence is not None and key != (career_key(evidence['university_name']),career_key(evidence['program_name_catalog'])):
            evidence=None
            row['warnings'].append('Program ID/name conflict: outcome enrichment rejected.')
        if evidence is not None and source.get('unitid') is not None and float(source['unitid'])!=float(evidence['unitid']):
            evidence=None
            row['warnings'].append('UNITID conflict: outcome enrichment rejected.')
        row.update({'outcome_evidence':None,'outcome_cluster':None,'earnings_context':None,
                    'school_placement_context':None,'research_industry_accuracy':'not_measured'})
        if evidence is None:
            row['warnings'].append('No exact uploaded program-outcome match; existing advisory fallback retained.')
            continue
        row['program_id']=evidence['program_id'];row['unitid']=int(evidence['unitid'])
        row['method']+=' + uploaded_outcomes_and_unsupervised_clusters'
        row['warnings']=[w.replace('Advisory rule result; no outcome-trained model is available.',
            'Career-path advice is heuristic; the separate trained model estimates cohort earnings.') for w in row['warnings']]
        row['outcome_evidence']={k:evidence.get(k) for k in ['row_type','estimate_method','evidence_level',
            'evidence_n','usable_for_scoring','thin_sample_flag','intl_student_applicability','credential_level',
            'outcome_source_unitid','unitid_source']}
        row['outcome_cluster']={'id':evidence.get('outcome_cluster'),
            'meaning':'Similarity of cohort earnings and debt; not a career-track prediction',
            'input_origin':evidence['estimate_method']}
        row['earnings_context']={'uploaded_4yr_earnings_usd':evidence.get('earnings_4yr'),
            'uploaded_estimate_method':evidence['estimate_method'],
            'model_4yr_earnings_usd':evidence['model_earnings_4yr'],
            'model_scope':'Field/credential cohort estimate; not personalized salary or placement probability',
            'currency':'USD','cohort_year':None,'international_applicability':'proxy only'}
        path=row['career_path']
        role_field='expected_roles_research' if path=='research' else 'expected_roles_industry' if path=='industry' else None
        if role_field and career_text(evidence.get(role_field)):
            row['expected_roles']=list(dict.fromkeys(r.strip() for r in evidence[role_field].split('|') if r.strip()))
            row['role_sources']=['Uploaded dataset: '+(evidence.get('roles_source') or 'curated CIP-family mapping')]
        # A financial/employment proxy can never fill avg_placement_stats.
        row['employment_proxy_context']={'value':evidence.get('employment_rate_proxy'),
            'type':'uploaded proxy, NOT a placement rate'}
        if (evidence.get('placement_source') and evidence.get('placement_rate') is not None
            and evidence.get('placement_year') is not None and int(evidence['placement_year']) <= reference_year):
            row['school_placement_context']={k:evidence.get(k) for k in ['placement_source','placement_rate',
                'placement_basis','placement_year','placement_level','placement_window_months','surveyed_count','placed_count']}
            row['school_placement_context']['verification']='as supplied; source page not independently checked in this run'
            row['warnings'].append('Placement evidence is school/cohort scoped and is not assigned to this specialization.')
        if not evidence.get('usable_for_scoring') or evidence.get('thin_sample_flag') or evidence['estimate_method']=='field_national_fallback':
            row['review_required']=True
            row['warnings'].append('Thin or restricted evidence: treat outcome context as provisional.')
        row['warnings'].append('No independent student research/industry labels; clustering does not establish track accuracy.')
    return result


career_agent=career_agent_v2
print('Active entry point: career_agent(state), enriched with the uploaded dataset and fitted outcome clusters.')



Active entry point: career_agent(state), enriched with the uploaded dataset and fitted outcome clusters.


## Executed handoff, catalog coverage and reload regression checks

In [12]:
new_demo=career_agent(agent3_live_demo)
assert agent3_live_demo==demo_original
assert [(r['university'],r['program_name']) for r in new_demo['careers']]==[
    (r['university'],r['program_name']) for r in agent3_live_demo['programs']]
new_coverage=career_agent(coverage_state)
catalog_counts_v2={'catalog_programs':len(new_coverage['careers']),
    'exact_outcome_matches':sum(r['outcome_evidence'] is not None for r in new_coverage['careers']),
    'with_cluster_assignment':sum(r['outcome_cluster'] is not None and r['outcome_cluster']['id'] is not None for r in new_coverage['careers']),
    'with_school_placement_context':sum(r['school_placement_context'] is not None for r in new_coverage['careers']),
    'with_program_placement_stats':sum(r['avg_placement_stats'] is not None for r in new_coverage['careers'])}
checks6=[]
def check6(name,condition):
    assert condition,name
    checks6.append({'check':name,'passed':True})
check6('input_state_preserved',agent3_live_demo==demo_original)
check6('all_catalog_programs_retained',len(new_coverage['careers'])==len(coverage_state['programs']))
check6('exact_mapping_coverage',catalog_counts_v2['exact_outcome_matches']==len(mapped_outcomes))
check6('proxies_not_reported_as_program_placement',catalog_counts_v2['with_program_placement_stats']==0)
check6('no_unseen_institution_overlap',not(set(observed.iloc[development_i].unitid)&set(observed.iloc[test_i].unitid)))
check6('estimates_excluded_from_training',observed.row_type.eq('scorecard_observed').all())
reload6=load_agent6_models()
with threadpool_limits(limits=1):
    check6('earnings_joblib_reload_identical',np.allclose(reload6['earnings'].predict(regression_inputs(reg_test.head(100))),test_earnings_predictions[:100]))
    check6('cluster_joblib_reload_identical',np.array_equal(reload6['clusters'].predict(cluster_test.head(100)),test_cluster_labels[:100]))
check6('lookup_reload_identical',reload6['lookup']['dataset_sha256']==data_fingerprint)
unknown=career_agent({'profile':{'career_goal':'industry'},'programs':[{'university':'Unknown university','program_name':'Unknown program'}]})['careers'][0]
check6('unknown_program_no_fabricated_outcomes',unknown['outcome_evidence'] is None and unknown['earnings_context'] is None)
conflict=deepcopy(agent3_live_demo)
conflict['programs'][0]['program_id']=mapped_outcomes.iloc[0].program_id
conflict_result=career_agent(conflict)['careers'][0]
check6('conflicting_program_identity_rejected',conflict_result['outcome_evidence'] is None)
check6('no_supervised_track_classifier_fabricated',all(r['research_industry_accuracy']=='not_measured' for r in new_coverage['careers']))
mba=next(r for r in agent6_models['lookup']['programs'] if r.get('placement_rate') is not None)
historical=career_agent({'profile':{'career_goal':'industry'},'programs':[
    {'university':mba['university_name'],'program_name':mba['program_name_catalog']}]},as_of='2000-01-01')
check6('future_placement_cohort_withheld',historical['careers'][0]['school_placement_context'] is None)
print('Checks:',len(checks6),'passed')
print('Catalog coverage:',catalog_counts_v2)
print('Actual Agent 3 -> Agent 6 result:')
print(json.dumps(new_demo['careers'],indent=2))
(CAREER_REPORT_DIR/'demo_predictions.json').write_text(json.dumps(new_demo['careers'],indent=2),encoding='utf-8')



Checks: 13 passed
Catalog coverage: {'catalog_programs': 3052, 'exact_outcome_matches': 3033, 'with_cluster_assignment': 3021, 'with_school_placement_context': 41, 'with_program_placement_stats': 0}
Actual Agent 3 -> Agent 6 result:
[
  {
    "university": "Stanford University",
    "program_name": "MS Computer Science - Systems",
    "country": "United States",
    "career_path": "industry",
    "alignment_score": 0.6666666666666666,
    "expected_roles": [
      "Software Engineer",
      "Backend/Full-Stack Developer",
      "Machine Learning Engineer"
    ],
    "avg_placement_stats": null,
    "method": "rule_based_advisory + uploaded_outcomes_and_unsupervised_clusters",
    "alignment_score_type": "heuristic_goal_evidence_share",
    "model_confidence": null,
    "stated_goal": "industry",
    "goal_interpretation": "explicit career_goal",
    "track_evidence": {
      "research": [
        "research"
      ],
      "industry": [
        "engineering",
        "software"
      ]


## Final executed metrics and artifact manifest

In [ ]:
manifest6={p.name:{'sha256':hashlib.sha256(p.read_bytes()).hexdigest(),'bytes':p.stat().st_size}
           for p in CAREER_MODEL_DIR.glob('*.joblib')}
report6={'version':'agent6-outcomes-v2','created_utc':datetime.now(timezone.utc).isoformat(),
    'audit':audit6,'regression_test':regression_test_report,'baseline_test':baseline_test_report,
    'clustering_test':cluster_test_report,'clustering_dev_candidates':cluster_candidates,
    'catalog_coverage':catalog_counts_v2,'functional_checks':checks6,'artifacts':manifest6,
    'sklearn_version':sklearn.__version__,'research_industry_classification_accuracy':None,
    'note':'Earnings metrics compare held-out uploaded cohort figures. Clustering metrics are not accuracy. Source rows and years are not independently reconstructed.'}
(CAREER_REPORT_DIR/'validation_report.json').write_text(json.dumps(report6,indent=2),encoding='utf-8')
(CAREER_MODEL_DIR/'manifest.json').write_text(json.dumps({'artifacts':manifest6,'dataset_sha256':data_fingerprint,
    'sklearn_version':sklearn.__version__,'regression_features':REG_FEATURES,'cluster_features':CLUSTER_FEATURES},indent=2),encoding='utf-8')
print('AGENT 6 — FINAL EXECUTED RESULTS')
print(f"Uploaded data: {len(outcomes):,} rows; exact program matches: {catalog_counts_v2['exact_outcome_matches']}/{catalog_counts_v2['catalog_programs']}")
print(f"Held-out earnings MAE: ${regression_test_report['mae_usd']:,.2f}; median baseline MAE: ${baseline_test_report['mae_usd']:,.2f}")
print(f"Held-out earnings R²: {regression_test_report['r2']:.4f}; share within ±10%: {regression_test_report['within_10_percent']:.2%}")
print(f"Unsupervised clusters: {best_k}; held-out silhouette: {cluster_test_report['silhouette']:.4f}; Davies–Bouldin: {cluster_test_report['davies_bouldin']:.4f}")
print(f"Functional checks: {len(checks6)}/{len(checks6)}; all three joblib artifacts saved and reloaded successfully.")
print('Research-versus-industry accuracy: NOT MEASURED — no independent track labels in uploaded data.')
print('Joblib directory:',CAREER_MODEL_DIR)


AGENT 6 — FINAL EXECUTED RESULTS
Uploaded data: 51,842 rows; exact program matches: 3033/3052
Held-out earnings MAE: $10,321.84; median baseline MAE: $19,150.85
Held-out earnings R²: 0.6343; share within ±10%: 48.18%
Unsupervised clusters: 2; held-out silhouette: 0.4134; Davies–Bouldin: 1.0153
Functional checks: 13/13; all three joblib artifacts saved and reloaded successfully.
Research-versus-industry accuracy: NOT MEASURED — no independent track labels in uploaded data.
Cluster quality and earnings-regression scores are not a 96% career-placement accuracy claim.
Joblib directory: D:\Users\asmita\MAJOR\Major_Project-2027\backend\app\agents\career_agent\trained_models
